In [17]:
import pandas as pd
Phishing_Data = pd.read_csv("C:\\Users\\alfre_g2qn6y7\\OneDrive\\Documents\\dataset_phishing.csv")
Phishing_Data

,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11425,http://www.fontspace.com/category/blackletter,45,17,0,2,0,0,0,0,0,...,0,0,0,448,5396,3980,0,0,6,legitimate
11426,http://www.budgetbots.com/server.php/Server%20...,84,18,0,5,0,1,1,0,0,...,1,0,0,211,6728,0,0,1,0,phishing
11427,https://www.facebook.com/Interactive-Televisio...,105,16,1,2,6,0,1,0,0,...,0,0,0,2809,8515,8,0,1,10,legitimate
11428,http://www.mypublicdomainpictures.com/,38,30,0,2,0,0,0,0,0,...,1,0,0,85,2836,2455493,0,0,4,legitimate


In [18]:
X = Phishing_Data.drop(["url","status"],axis=1)
y = Phishing_Data["status"]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)


In [21]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import BernoulliNB
model = BernoulliNB()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test,y_pred)
print(accuracy)

0.8801399825021873


In [22]:
report = classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

  legitimate       0.87      0.90      0.88      1157
    phishing       0.89      0.86      0.88      1129

    accuracy                           0.88      2286
   macro avg       0.88      0.88      0.88      2286
weighted avg       0.88      0.88      0.88      2286



In [23]:
from sklearn.model_selection import KFold, cross_val_score
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
scores.mean()

0.8798123593502686

Results are good but not as good as that of the tree models or standardized linear models. No disparity between f1 scores for legitimate and phishing samples. Let's do hyperparameter tuning now. 

In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics  import accuracy_score, classification_report
bnb = BernoulliNB()
param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    'fit_prior': [True, False] # Whether to learn class prior probabilities or assume uniform
}
grid_search = GridSearchCV(
    estimator=bnb, 
    param_grid=param_grid, 
    scoring='accuracy', # Metric to optimize (e.g., accuracy, f1, recall, etc.)
    cv=5, 
    verbose=1, 
    n_jobs=-1 # Use all available CPU cores
)
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)
print("Grid Search complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.3f}")
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy of the best model: {test_accuracy:.3f}")
report = classification_report(y_test,y_pred)
print(report)


Starting Grid Search...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Grid Search complete.
Best parameters found: {'alpha': 0.01, 'fit_prior': True}
Best cross-validation accuracy: 0.881
Test accuracy of the best model: 0.880
              precision    recall  f1-score   support

  legitimate       0.87      0.90      0.88      1157
    phishing       0.89      0.86      0.88      1129

    accuracy                           0.88      2286
   macro avg       0.88      0.88      0.88      2286
weighted avg       0.88      0.88      0.88      2286



Some improvement in cross validation score but largely same results as above. 

In [24]:
import pandas as pd
Email = pd.read_csv("C:\\Users\\alfre_g2qn6y7\\OneDrive\\Documents\\Phishing_Email.csv")
Email

,Unnamed: 0,Email Text,Email Type,num_chars,num_words,avg_word_len,num_sentences,num_capitals,num_exclamations,num_question_marks,num_special_chars,num_digits,num_urls
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,1030.0,230.0,3.482609,9.0,0.0,2.0,0.0,59.0,9.0,0.0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,479.0,91.0,4.274725,6.0,0.0,0.0,2.0,16.0,0.0,0.0
2,2,re : equistar deal tickets are you still avail...,Safe Email,1245.0,305.0,3.085246,7.0,0.0,0.0,1.0,95.0,63.0,0.0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,688.0,96.0,5.500000,38.0,39.0,1.0,1.0,110.0,29.0,1.0
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,441.0,91.0,3.857143,13.0,0.0,0.0,0.0,27.0,2.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17711,18646,date a lonely housewife always wanted to date ...,Phishing Email,237.0,45.0,4.288889,6.0,0.0,0.0,1.0,8.0,0.0,0.0
17712,18647,request submitted : access request for anita ....,Safe Email,477.0,99.0,3.828283,8.0,0.0,0.0,0.0,31.0,24.0,0.0
17713,18648,"re : important - prc mtg hi dorn & john , as y...",Safe Email,1214.0,253.0,3.802372,13.0,0.0,1.0,1.0,38.0,0.0,0.0
17714,18649,press clippings - letter on californian utilit...,Safe Email,213.0,34.0,5.294118,0.0,0.0,0.0,0.0,8.0,0.0,0.0


In [25]:
X = Email.drop(["Unnamed: 0","Email Text","Email Type"],axis=1)
y = Email["Email Type"]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [10]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import BernoulliNB
model = BernoulliNB()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test,y_pred)
print(accuracy)

0.7364559819413092


In [11]:
report = classification_report(y_test,y_pred)
print(report)

                precision    recall  f1-score   support

Phishing Email       0.73      0.55      0.63      1429
    Safe Email       0.74      0.86      0.80      2115

      accuracy                           0.74      3544
     macro avg       0.73      0.71      0.71      3544
  weighted avg       0.74      0.74      0.73      3544



In [12]:
from sklearn.model_selection import KFold, cross_val_score
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
scores.mean()

0.7502811538897166

Results are worse than for the first dataset. There is a significant disparity between f1 scores for both classes. Let's do hyperparameter tuning now. 

In [26]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics  import accuracy_score, classification_report
bnb = BernoulliNB()
param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    'fit_prior': [True, False] # Whether to learn class prior probabilities or assume uniform
}
grid_search = GridSearchCV(
    estimator=bnb, 
    param_grid=param_grid, 
    scoring='accuracy', # Metric to optimize (e.g., accuracy, f1, recall, etc.)
    cv=5, 
    verbose=1, 
    n_jobs=-1 # Use all available CPU cores
)
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)
print("Grid Search complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.3f}")
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy of the best model: {test_accuracy:.3f}")
report = classification_report(y_test,y_pred)
print(report)

Starting Grid Search...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Grid Search complete.
Best parameters found: {'alpha': 0.01, 'fit_prior': True}
Best cross-validation accuracy: 0.750
Test accuracy of the best model: 0.736
                precision    recall  f1-score   support

Phishing Email       0.73      0.55      0.63      1429
    Safe Email       0.74      0.86      0.80      2115

      accuracy                           0.74      3544
     macro avg       0.73      0.71      0.71      3544
  weighted avg       0.74      0.74      0.73      3544



In [ ]:
No real improvement. 

In [27]:
URL = pd.read_csv("C:\\Users\\alfre_g2qn6y7\\OneDrive\\Documents\\web-page-phishing.csv")
URL

,url_length,n_dots,n_hypens,n_underline,n_slash,n_questionmark,n_equal,n_at,n_and,n_exclamation,n_space,n_tilde,n_comma,n_plus,n_asterisk,n_hastag,n_dollar,n_percent,n_redirection,phishing
0,37,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,77,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
2,126,4,1,2,0,1,3,0,2,0,0,0,0,0,0,0,0,0,1,1
3,18,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,55,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100072,23,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
100073,34,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0
100074,70,2,1,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
100075,28,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [28]:
X = URL.drop(["phishing"],axis=1)
y = URL["phishing"]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [29]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import BernoulliNB
model = BernoulliNB()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test,y_pred)
print(accuracy)

0.8247901678657075


In [30]:
report = classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.87      0.86      0.86     12698
           1       0.76      0.77      0.76      7318

    accuracy                           0.82     20016
   macro avg       0.81      0.81      0.81     20016
weighted avg       0.83      0.82      0.83     20016



In [31]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
scores.mean()

0.8223104918859722

Test and cross validation accuracy results are higher than for the second dataset. Non-phishing urls have a much higher f1 score than phishing urls.  

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics  import accuracy_score, classification_report
bnb = BernoulliNB()
param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    'fit_prior': [True, False] # Whether to learn class prior probabilities or assume uniform
}
grid_search = GridSearchCV(
    estimator=bnb, 
    param_grid=param_grid, 
    scoring='accuracy', # Metric to optimize (e.g., accuracy, f1, recall, etc.)
    cv=5, 
    verbose=1, 
    n_jobs=-1 # Use all available CPU cores
)
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)
print("Grid Search complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.3f}")
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy of the best model: {test_accuracy:.3f}")
report = classification_report(y_test,y_pred)
print(report)

Starting Grid Search...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Grid Search complete.
Best parameters found: {'alpha': 10.0, 'fit_prior': False}
Best cross-validation accuracy: 0.835
Test accuracy of the best model: 0.838
              precision    recall  f1-score   support

           0       0.92      0.82      0.86     12698
           1       0.73      0.87      0.80      7318

    accuracy                           0.84     20016
   macro avg       0.83      0.85      0.83     20016
weighted avg       0.85      0.84      0.84     20016



Better test and cross validation results than pre-hyperparameter tuning. The f1 score for phishing urls is significantly better. 